# Bronze ingestion

Downloads GitHub Copilot report payloads to the attached lakehouse before any Spark read. The package and its dependencies must be installed in the Fabric Environment attached to this notebook.

In [ ]:
# ruff: noqa: E501, F821
# Pipeline parameters. Keep customer identifiers and secret values outside source control.
run_id = "manual"
run_mode = "daily"  # daily or backfill
scope_kind = "organization"  # organization or enterprise
scope_slug = ""
entity_type = "copilot_usage"
report_types = "entity,users,user-teams,repositories"
trailing_days = 28
start_date = ""
end_date = ""
replacement_start_date = ""  # Gold output start; earlier complete dates may be reused
lakehouse_files_root = "/lakehouse/default/Files"
bronze_folder = "bronze"
audit_schema = "audit"
key_vault_uri = ""
github_token_secret_name = ""


In [ ]:
import json
from datetime import date, datetime, timezone
from pathlib import Path

from notebookutils import mssparkutils
from pyspark.sql.types import StringType, StructField, StructType

from copilot_metrics_fabric.bronze import (
    BronzeBatchError,
    BronzeIngestor,
    LocalBronzeStorage,
    report_days,
)
from copilot_metrics_fabric.config import GitHubConfig
from copilot_metrics_fabric.github_client import GitHubCopilotClient, ReportType

VALID_REPORT_TYPES = {item.value: item for item in ReportType}
AUDIT_COLUMNS = ("run_id", "stage", "status", "started_at", "completed_at", "scope_kind", "scope_slug", "start_date", "end_date", "validation_status", "details_json")

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def write_audit(status: str, started_at: str, validation_status: str, details: dict) -> None:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{audit_schema}`")
    schema = StructType([StructField(name, StringType(), False) for name in AUDIT_COLUMNS])
    row = (run_id, "bronze", status, started_at, utc_now(), scope_kind, scope_slug, effective_start, effective_end, validation_status, json.dumps(details, sort_keys=True))
    spark.createDataFrame([row], schema).write.format("delta").mode("append").saveAsTable(f"{audit_schema}.pipeline_run_results")


In [ ]:
started_at = utc_now()
effective_start = start_date
effective_end = end_date

try:
    if not scope_slug.strip():
        raise ValueError("scope_slug must be supplied by the pipeline or operator")
    if scope_kind not in {"organization", "enterprise"}:
        raise ValueError("scope_kind must be organization or enterprise")
    if run_mode not in {"daily", "backfill"}:
        raise ValueError("run_mode must be daily or backfill")
    selected_report_types = tuple(VALID_REPORT_TYPES[name.strip()] for name in report_types.split(",") if name.strip())
    if not selected_report_types:
        raise ValueError("report_types must select at least one supported report")

    if start_date and end_date:
        days = report_days(start_day=date.fromisoformat(start_date), end_day=date.fromisoformat(end_date))
    elif run_mode == "backfill":
        raise ValueError("backfill runs require start_date and end_date")
    else:
        days = report_days(trailing_days=int(trailing_days))
    reuse_before = (
        date.fromisoformat(replacement_start_date)
        if replacement_start_date
        else None
    )
    effective_start = days[0].isoformat()
    effective_end = days[-1].isoformat()

    if not key_vault_uri or not github_token_secret_name:
        raise ValueError("key_vault_uri and github_token_secret_name must be supplied externally")
    token = mssparkutils.credentials.getSecret(key_vault_uri, github_token_secret_name)
    github_config = GitHubConfig(
        mode=scope_kind,
        organizations=(scope_slug,) if scope_kind == "organization" else (),
        enterprise=scope_slug if scope_kind == "enterprise" else None,
    )
    client = GitHubCopilotClient(github_config, lambda: token)

    # External HTTPS payloads are streamed to lakehouse Files by the Bronze contract.
    # Spark is deliberately not used until every completed download has landed.
    bronze_root = Path(lakehouse_files_root) / bronze_folder
    ingestor = BronzeIngestor(client, LocalBronzeStorage(bronze_root))
    completed = []
    failures = []
    try:
        completed.extend(ingestor.ingest(days, selected_report_types, reuse_before=reuse_before))
    except BronzeBatchError as error:
        completed.extend(error.completed)
        failures.extend(error.failures)

    landed_files = [str(bronze_root / item.path) for result in completed if result.status != "reused" for item in result.files]
    expected_records = sum(item.record_count for result in completed if result.status != "reused" for item in result.files)
    # This is the first Spark read: it targets only already-landed lakehouse files.
    observed_records = spark.read.text(landed_files).count() if landed_files else 0
    if observed_records != expected_records:
        raise RuntimeError(f"landed record validation failed: expected={expected_records}, observed={observed_records}")
    if failures:
        raise RuntimeError(f"{len(failures)} Bronze report request(s) failed")

    details = {
        "entity_type": entity_type,
        "requests": len(completed),
        "landed_files": len(landed_files),
        "landed_records": observed_records,
        "reused_requests": sum(result.status == "reused" for result in completed),
        "statuses": sorted({result.status for result in completed}),
    }
    write_audit("succeeded", started_at, "passed", details)
    mssparkutils.notebook.exit(json.dumps(details, sort_keys=True))
except Exception as error:
    safe_details = {"entity_type": entity_type, "error_type": type(error).__name__}
    try:
        write_audit("failed", started_at, "failed", safe_details)
    finally:
        raise
